In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if cwd.name == 'url_corpus' and cwd.parent.name == 'notebooks':
    os.chdir(cwd.parent)

nb_dir = Path.cwd().resolve()
backend_root = nb_dir.parent if nb_dir.name == 'notebooks' else nb_dir
if str(backend_root) not in sys.path:
    sys.path.insert(0, str(backend_root))

print(f'Notebook CWD: {Path.cwd()}')
print(f'Backend root in sys.path: {backend_root}')


# Analiza Nierownowagi Klas: Zbior Zbalansowany vs Niezbalansowany

Ten notebook porownuje zachowanie modelu dla: oryginalnego zbioru niezbalansowanego, niedoprobkowania, nadprobkowania oraz syntetycznego nadprobkowania (SMOTE).


## Metodologia

1. Podziel dane ze stratyfikacja.
2. Pozostaw zbior testowy bez zmian (oryginalny rozklad klas).
3. Zbalansuj tylko dane treningowe.
4. Trenuj ten sam klasyfikator dla kazdego wariantu.
5. Porownaj PR-AUC, F1 klasy mniejszosciowej, Czulosc klasy mniejszosciowej, Zrownowazona trafnosc i macierz pomylek.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    f1_score,
    recall_score,
    balanced_accuracy_score,
    confusion_matrix
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE

In [ ]:
DATA_PATH = '../data/url_corpus/dataset_full.csv'
TARGET_COL = 'phishing'
TEST_SIZE = 0.2
RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH)
print(f'Rozmiar zbioru: {df.shape}')
print('Kolumny:', list(df.columns)[:10], '...')

if TARGET_COL not in df.columns:
    candidates = ['phishing', 'label', 'target', 'class', 'y']
    detected = next((c for c in candidates if c in df.columns), None)
    if detected is None:
        raise ValueError(f"Nie znaleziono kolumny docelowej '{TARGET_COL}'. Dostepne kolumny: {list(df.columns)}")
    TARGET_COL = detected
    print(f"TARGET_COL ustawiono automatycznie na: {TARGET_COL}")

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

class_counts = y.value_counts(dropna=False).sort_index()
print('\nRozklad klas (pelny zbior):')
print(class_counts)
imbalance_ratio = class_counts.max() / class_counts.min()
print(f'Wspolczynnik niezbalansowania (major/minor): {imbalance_ratio:.3f}')

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(x=y)
plt.title('Rozklad klas w pelnym zbiorze')
plt.xlabel('Klasa')
plt.ylabel('Liczba probek')
plt.tight_layout()
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

print('Rozmiar zbioru treningowego:', X_train.shape, 'Rozmiar zbioru testowego:', X_test.shape)
print('Rozklad klas w zbiorze treningowym:')
print(y_train.value_counts().sort_index())
print('Rozklad klas w zbiorze testowym:')
print(y_test.value_counts().sort_index())

## Znaczenie wariantow balansowania

- `bez balansowania`: dane treningowe bez zmian.
- `niedoprobkowanie`: zmniejszenie liczby probek klasy wiekszosciowej.
- `nadprobkowanie`: zwiekszenie liczby probek klasy mniejszosciowej przez duplikacje.
- `syntetyczne_nadprobkowanie`: syntetyczne nadprobkowanie (SMOTE), czyli tworzenie nowych probek klasy mniejszosciowej.

Praktyczna interpretacja:

- niedoprobkowanie: szybszy trening, ale ryzyko utraty informacji z klasy wiekszosciowej,
- nadprobkowanie: brak utraty danych klasy wiekszosciowej, ale ryzyko nadmiernego dopasowania,
- syntetyczne nadprobkowanie (SMOTE): zwykle lepszy kompromis miedzy recall a precision dla klasy mniejszosciowej.


In [ ]:
pipelines = {
    'bez balansowania': SkPipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
    ], memory=None),
    'niedoprobkowanie': ImbPipeline([
        ('sampler', RandomUnderSampler(random_state=RANDOM_STATE)),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
    ], memory=None),
    'nadprobkowanie': ImbPipeline([
        ('sampler', RandomOverSampler(random_state=RANDOM_STATE)),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
    ], memory=None),
    'syntetyczne_nadprobkowanie': ImbPipeline([
        ('sampler', SMOTE(random_state=RANDOM_STATE)),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
    ], memory=None),
}

minority_label = y.value_counts().idxmin()
print('Wykryta klasa mniejszosciowa:', minority_label)

In [ ]:
results = []
pr_curves = {}
cms = {}

for variant, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    if hasattr(pipe, 'predict_proba') and len(np.unique(y_test)) == 2:
        y_prob = pipe.predict_proba(X_test)[:, 1]
        pr_auc = average_precision_score(y_test, y_prob)
        precision, recall, _ = precision_recall_curve(y_test, y_prob, pos_label=minority_label)
        pr_curves[variant] = (recall, precision)
    else:
        y_prob = None
        pr_auc = np.nan

    f1_min = f1_score(y_test, y_pred, pos_label=minority_label)
    rec_min = recall_score(y_test, y_pred, pos_label=minority_label)
    bacc = balanced_accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    cms[variant] = cm

    results.append({
        'wariant': variant,
        'pr_auc': pr_auc,
        'f1_mniejszosc': f1_min,
        'czulosc_mniejszosci': rec_min,
        'zrownowazona_trafnosc': bacc
    })

results_df = pd.DataFrame(results).sort_values(by='pr_auc', ascending=False)
results_df

In [ ]:
if pr_curves:
    plt.figure(figsize=(8, 6))
    for variant, (recall, precision) in pr_curves.items():
        auc_val = results_df.loc[results_df['wariant'] == variant, 'pr_auc'].iloc[0]
        plt.plot(recall, precision, label=f"{variant} (AP={auc_val:.3f})")

    plt.xlabel('Czulosc (Recall)')
    plt.ylabel('Precyzja (Precision)')
    plt.title('Krzywe Precision-Recall dla wariantow zbioru')
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print('Krzywe PR niedostepne dla obecnego formatu etykiety.')

In [ ]:
n = len(cms)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
if n == 1:
    axes = [axes]

for ax, (variant, cm) in zip(axes, cms.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    ax.set_title(variant)
    ax.set_xlabel('Predykcja')
    ax.set_ylabel('Wartosc rzeczywista')

plt.suptitle('Macierze pomylek na tym samym niezmienionym zbiorze testowym', y=1.05)
plt.tight_layout()
plt.show()

## Wnioski interpretacyjne

- Jesli balansowanie podnosi czulosc klasy mniejszosciowej, ale obniza precyzje, liczba falszywych alarmow rosnie.
- Dla detekcji phishingu priorytetem zwykle jest wysoka czulosc klasy mniejszosciowej (podejscie security-first).
- Model oceniaj lacznie przez PR-AUC i czulosc klasy mniejszosciowej, nie tylko przez trafnosc (accuracy).

## Gotowy opis wynikow do pracy dyplomowej

W badaniu porownano cztery warianty danych treningowych: bez balansowania, niedoprobkowanie, nadprobkowanie oraz syntetyczne nadprobkowanie (SMOTE). Wszystkie modele oceniono na tym samym, niezmienionym zbiorze testowym.

Najwyzsze PR-AUC uzyskano dla wariantu: [UZUPELNIJ].
Najwyzsza czulosc klasy mniejszosciowej wystapila dla wariantu: [UZUPELNIJ].
Najwyzszy wynik F1 dla klasy mniejszosciowej uzyskano dla wariantu: [UZUPELNIJ].

Wyniki wskazuja, ze zastosowanie [UZUPELNIJ: niedoprobkowanie/nadprobkowanie/syntetyczne nadprobkowanie (SMOTE)] poprawia wykrywanie klasy phishing (klasy mniejszosciowej) kosztem [UZUPELNIJ: spadku precyzji / wzrostu liczby falszywych alarmow].

W kontekscie systemu detekcji phishingu priorytetem jest ograniczenie liczby przypadkow niewykrytego phishingu, dlatego kluczowa metryka jest czulosc klasy mniejszosciowej oraz PR-AUC.
